# Exploratory data analysis

What the training data looks like, for the Data slide (course brief: source, size, key
properties, cleaning). Runs on Colab, which already has pandas and matplotlib; the project's
Docker image doesn't need them.

It reads the same data pack as `colab_train.ipynb` (`MyDrive/battlemap-walls/packs/`): the
manifest for every map, and the 32 px images and masks of the train and val splits. Test maps
never leave the laptop, so the test split appears only as counts and pixel shares from the
manifest. Every figure is saved to `MyDrive/battlemap-walls/eda/` as a PNG for the slides.

In [ ]:
# Settings
REPO = "https://github.com/Darkanath/huji-ai-engineering-battlemap-walls.git"
COMMIT = "main"
PACK = ""  # file name in MyDrive/battlemap-walls/packs/; "" = the newest pack

DRIVE = "/content/drive/MyDrive/battlemap-walls"
PROCESSED = "/content/pack"  # not /content/data: that would shadow the `data` package
CODE = "/content/battlemap-walls"
FIGURES = f"{DRIVE}/eda"

## Setup: Drive, code, data pack

In [ ]:
import glob
import os
import subprocess
import zipfile

from google.colab import drive

drive.mount("/content/drive")
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--quiet", url, CODE], check=True)
subprocess.run(["git", "-C", CODE, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "--quiet", COMMIT], check=True)
if COMMIT == "main":
    subprocess.run(["git", "-C", CODE, "pull", "--quiet", "origin", "main"], check=True)

packs = sorted(glob.glob(f"{DRIVE}/packs/data-*.zip"), key=os.path.getmtime)
pack = f"{DRIVE}/packs/{PACK}" if PACK else packs[-1]
os.makedirs(PROCESSED, exist_ok=True)
with zipfile.ZipFile(pack) as zf:
    zf.extractall(PROCESSED)
os.makedirs(FIGURES, exist_ok=True)
print("pack:", pack)

In [ ]:
import sys

# The editable install only takes effect after a kernel restart, so also import the project
# straight from the clone.
%pip install --quiet --no-deps -e {CODE}
os.chdir(CODE)
sys.path.insert(0, CODE)

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from data.rasterize import CLASS_NAMES, LIGHT_TIERS, PORTAL, overlay

DATASETS = ["synthetic", "dungeondraft", "dungeon_alchemist"]
COLORS = {"synthetic": "#7a7a7a", "dungeondraft": "#c0392b", "dungeon_alchemist": "#2471a3"}
PPG = 32  # the training copies are at 32 px per square


def save(fig, name):
    """Save a figure for the slides and show it."""
    fig.savefig(f"{FIGURES}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


df = pd.read_csv(Path(PROCESSED) / "manifest.csv")
df = df[df["split"] != "fixture"]
df["area"] = df["width"] * df["height"]
local = df[df["split"].isin(["train", "val"])].copy()  # maps whose images are in the pack
print(len(df), "maps in the manifest;", len(local), "train/val maps in the pack")

## 1. Size: maps and layouts per dataset and split

Splits are by layout: all style variants of one layout are in the same split.

In [ ]:
maps = df.pivot_table(index="dataset", columns="split", values="map", aggfunc="count", fill_value=0)
layouts = df.pivot_table(
    index="dataset", columns="split", values="layout", aggfunc="nunique", fill_value=0
)
summary = maps.astype(str) + " maps / " + layouts.astype(str) + " layouts"
df["size"] = df["width"].astype(str) + "x" + df["height"].astype(str)
summary["map size (squares)"] = df.groupby("dataset")["size"].agg(
    lambda s: ", ".join(sorted(set(s)))
)
summary

## 2. Class balance

Share of pixels per class, weighting each map by its area. This is why the loss uses class
weights and the crops are biased toward walls, portals and lights (docs/training.md §2, §4).

In [ ]:
share_cols = [f"share_{name}" for name in CLASS_NAMES.values()]
weighted = df[share_cols].mul(df["area"], axis=0).groupby(df["dataset"]).sum()
shares = weighted.div(df.groupby("dataset")["area"].sum(), axis=0)
shares.columns = list(CLASS_NAMES.values())
display((shares * 100).round(2).astype(str) + " %")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, name in zip(axes, ["wall", "portal", "light"], strict=True):
    data = [df.loc[df["dataset"] == d, f"share_{name}"] * 100 for d in DATASETS]
    ax.boxplot(data, tick_labels=DATASETS)
    ax.set_title(f"{name} pixels per map (%)")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Class share per map, by dataset")
save(fig, "class_shares")

## 3. Lights per size tier

A light's tier comes from its `range` in the .dd2vtt (docs/labeling-rules.md). Maps from before
lights were added (`dd01`–`dd03`) have none.

In [ ]:
tiers = [f"lights_{t}" for t in LIGHT_TIERS]
lights = df.groupby("dataset")[tiers].sum()
lights.columns = list(LIGHT_TIERS)
lights["maps with lights"] = df.groupby("dataset").apply(
    lambda g: int((g[tiers].sum(axis=1) > 0).sum())
)
display(lights)

fig, ax = plt.subplots(figsize=(6, 3.5))
lights[list(LIGHT_TIERS)].plot.bar(ax=ax, color=["#f4d03f", "#e67e22", "#c0392b"])
ax.set_ylabel("lights")
ax.set_title("Lights per tier")
ax.tick_params(axis="x", rotation=0)
save(fig, "lights_per_tier")

## 4. Portal widths

Each portal blob in the 32 px masks, measured along its long side, in squares. Double doors in
Dungeon Alchemist are two 1-square portals side by side, so they merge into one 2-square blob.

In [ ]:
def portal_widths(mask):
    """Long side of each portal blob, in squares."""
    n, labels = cv2.connectedComponents((mask == PORTAL).astype(np.uint8))
    widths = []
    for i in range(1, n):
        pts = np.column_stack(np.nonzero(labels == i)).astype(np.float32)
        (_, _), (w, h), _ = cv2.minAreaRect(pts)
        widths.append(max(w, h) / PPG)
    return widths


widths = {d: [] for d in DATASETS}
for row in local.itertuples():
    mask = np.array(Image.open(Path(PROCESSED) / row.mask_32))
    widths[row.dataset] += portal_widths(mask)

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.arange(0, 5.25, 0.25)
for d in DATASETS:
    if widths[d]:
        weights = np.full(len(widths[d]), 1 / len(widths[d]))
        ax.hist(widths[d], bins=bins, weights=weights, alpha=0.6, label=d, color=COLORS[d])
ax.set_xlabel("portal width (squares)")
ax.set_ylabel("share of portals")
ax.legend()
ax.set_title("Portal widths (train and val maps)")
save(fig, "portal_widths")

## 5. How different the datasets look

The synthetic maps are for pretraining; the model is fine-tuned on the real maps (Dungeondraft
and Dungeon Alchemist). These plots show how far apart they are in color and brightness, which
is part of why stage 2 fine-tuning is needed.

In [ ]:
stats = []
hists = {d: np.zeros((3, 32)) for d in DATASETS}
for row in local.itertuples():
    image = np.array(Image.open(Path(PROCESSED) / row.image_32).convert("RGB"))
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    stats.append(
        {"dataset": row.dataset, "brightness": hsv[..., 2].mean(), "saturation": hsv[..., 1].mean()}
    )
    for c in range(3):
        hists[row.dataset][c] += np.histogram(image[..., c], bins=32, range=(0, 256))[0]
stats = pd.DataFrame(stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for d in DATASETS:
    s = stats[stats["dataset"] == d]
    axes[0].scatter(s["brightness"], s["saturation"], s=10, alpha=0.6, label=d, color=COLORS[d])
axes[0].set_xlabel("mean brightness (V)")
axes[0].set_ylabel("mean saturation (S)")
axes[0].legend()
axes[0].set_title("One point per map")
centers = np.arange(4, 256, 8)
for d in DATASETS:
    if hists[d].sum():
        gray = hists[d].sum(0) / hists[d].sum()
        axes[1].plot(centers, gray, label=d, color=COLORS[d])
axes[1].set_xlabel("pixel value (all channels)")
axes[1].set_ylabel("share of pixels")
axes[1].legend()
axes[1].set_title("Pixel value histogram per dataset")
save(fig, "domain_gap")

## 6. Examples with labels

Random 256 px crops (8×8 squares) from each dataset, with walls (red), portals (magenta) and
lights (yellow) tinted, as the model sees them before augmentation.

In [ ]:
rng = np.random.default_rng(0)
per_dataset = 4
present = [d for d in DATASETS if (local["dataset"] == d).any()]
fig, axes = plt.subplots(len(present), per_dataset, figsize=(3 * per_dataset, 3 * len(present)))
axes = np.atleast_2d(axes)
for r, d in enumerate(present):
    rows = local[local["dataset"] == d].sample(per_dataset, replace=True, random_state=r)
    for c, row in enumerate(rows.itertuples()):
        image = Image.open(Path(PROCESSED) / row.image_32).convert("RGB")
        mask = np.array(Image.open(Path(PROCESSED) / row.mask_32))
        top = rng.integers(0, max(1, mask.shape[0] - 256))
        left = rng.integers(0, max(1, mask.shape[1] - 256))
        box = (left, top, left + 256, top + 256)
        axes[r, c].imshow(overlay(image.crop(box), mask[top : top + 256, left : left + 256]))
        axes[r, c].set_title(row.map, fontsize=9)
        axes[r, c].axis("off")
fig.tight_layout()
save(fig, "examples")

## 7. Data cleaning

The checks every map passes before it is used are listed in PLAN.md ("Data cleaning & EDA"):
labels aligned with the image in the rasterizer preview, embedded image size = map size × px per
square, light ranges matching a tier, stacked lights merged, stacked portals flagged, stray
object shadows checked, and Dungeon Alchemist lights re-tiered with the original kept. Maps
dropped or fixed are listed with the reason in `docs/map-log.md`. Write the counts for the slide
here when the real data is complete.